# Tutorial 1: Learn about `requests`

Follow along with [this tutorial on RealPython](https://realpython.com/python-requests/). 
Complete the first 5 sections thoroughly (up to but not including User Other HTTP methods). Then jump to section Improve Performance to learn about some advanced tricks that you might need at some point when scraping websites that have a lot of content.

Use markdown headings as appropriate to enumerate sections.

## Inspect the response 

A Response is the object that contains the results of your request. Try making that same request again, but this time store the return value in a variable so you can get a closer look at its attributes and behaviors:


In [2]:
import requests
response = requests.get("https://api.github.com")

In this example, you’ve captured the return value of requests.get(). It’s an instance of Response, and you stored it in a variable called response. You can now use response to see a lot of information about the results of your GET request

## Work With Status Codes

A status code informs you of the status of the request.

a 200 OK status means that your request was successful, while a 404 NOT FOUND status means that the resource you were looking for wasn’t found.

In [3]:
response.status_code

200

Sometimes, you might want to use this information to make decisions in your code:

In [4]:
if response.status_code == 200:
    print("Success!")
elif response.status_code == 404:
    print("Not Found.")

Success!


Requests goes one step further in simplifying this process for you. If you use a Response instance in a Boolean context, such as a conditional statement, then it’ll evaluate to True when the status code is less than 400, and False otherwise.

That means you can modify the last example by rewriting the if statement:



In [5]:
if response:
    print("Success!")
else:
    raise Exception(f"Non-success status code: {response.status_code}")

Success!


n the code snippet above, you implicitly check whether the .status_code of response is between 200 and 399. If it’s not, then you raise an exception with an error message that includes the non-success status code wrapped in an f-string.

Keep in mind that this method does not verify whether the status code is equal to 200. This is because other status codes within the 200 to 399 range, such as 204 NO CONTENT and 304 NOT MODIFIED, are also considered successful because they provide some workable response.

In [6]:
import requests
from requests.exceptions import HTTPError

URLS = ["https://api.github.com", "https://api.github.com/invalid"]

for url in URLS:
    try:
        response = requests.get(url)
        response.raise_for_status()
    except HTTPError as http_err:
        print(f"HTTP error occurred: {http_err}")
    except Exception as err:
        print(f"Other error occurred: {err}")
    else:
        print("Success!")

Success!
HTTP error occurred: 404 Client Error: Not Found for url: https://api.github.com/invalid


## Access the Response Content

The response of a GET request often has some valuable information, known as a payload, in the message body. Using the attributes and methods of Response, you can view the payload in a variety of formats.

To see the response’s content in bytes, you use .content:


In [7]:
import requests

response = requests.get("https://api.github.com")
response.content


type(response.content)

bytes

While .content gives you access to the raw bytes of the response payload, you’ll often want to convert them into a string using a character encoding such as UTF-8. response will do that for you when you access .text:

In [8]:
response.text


type(response.text)

str

Because the decoding of bytes to a str requires an encoding scheme, Requests will try to guess the encoding based on the response’s headers if you don’t specify one. You can provide an explicit encoding by setting .encoding before accessing .text:

In [9]:
response.encoding = "utf-8"  # Optional: Requests infers this.
response.text

'{\n  "current_user_url": "https://api.github.com/user",\n  "current_user_authorizations_html_url": "https://github.com/settings/connections/applications{/client_id}",\n  "authorizations_url": "https://api.github.com/authorizations",\n  "code_search_url": "https://api.github.com/search/code?q={query}{&page,per_page,sort,order}",\n  "commit_search_url": "https://api.github.com/search/commits?q={query}{&page,per_page,sort,order}",\n  "emails_url": "https://api.github.com/user/emails",\n  "emojis_url": "https://api.github.com/emojis",\n  "events_url": "https://api.github.com/events",\n  "feeds_url": "https://api.github.com/feeds",\n  "followers_url": "https://api.github.com/user/followers",\n  "following_url": "https://api.github.com/user/following{/target}",\n  "gists_url": "https://api.github.com/gists{/gist_id}",\n  "hub_url": "https://api.github.com/hub",\n  "issue_search_url": "https://api.github.com/search/issues?q={query}{&page,per_page,sort,order}",\n  "issues_url": "https://api.g

In [19]:
response.json()

{'total_count': 33341016,
 'incomplete_results': False,
 'items': [{'id': 54346799,
   'node_id': 'MDEwOlJlcG9zaXRvcnk1NDM0Njc5OQ==',
   'name': 'public-apis',
   'full_name': 'public-apis/public-apis',
   'private': False,
   'owner': {'login': 'public-apis',
    'id': 51121562,
    'node_id': 'MDEyOk9yZ2FuaXphdGlvbjUxMTIxNTYy',
    'avatar_url': 'https://avatars.githubusercontent.com/u/51121562?v=4',
    'gravatar_id': '',
    'url': 'https://api.github.com/users/public-apis',
    'html_url': 'https://github.com/public-apis',
    'followers_url': 'https://api.github.com/users/public-apis/followers',
    'following_url': 'https://api.github.com/users/public-apis/following{/other_user}',
    'gists_url': 'https://api.github.com/users/public-apis/gists{/gist_id}',
    'starred_url': 'https://api.github.com/users/public-apis/starred{/owner}{/repo}',
    'subscriptions_url': 'https://api.github.com/users/public-apis/subscriptions',
    'organizations_url': 'https://api.github.com/users/pu

The type of the return value of .json() is a dictionary, so you can access values in the object by key:

In [ ]:
response_dict = response.json()

'https://api.github.com/emojis'

In [20]:
response_dict["emojis_url"]

'https://api.github.com/emojis'

## View Response Headers

The response headers can give you useful information, such as the content type of the response payload and how long to cache the response. To view these headers, access .headers:

In [11]:
import requests

response = requests.get("https://api.github.com")
response.headers

{'Date': 'Wed, 09 Sep 2026 17:12:12 GMT', 'Cache-Control': 'public, max-age=60, s-maxage=60', 'Vary': 'Accept,Accept-Encoding, Accept, X-Requested-With', 'ETag': '"4f825cc84e1c733059d46e76e6df9db557ae5254f9625dfe8e1b09499c449438"', 'x-github-api-version-selected': '2022-11-28', 'Access-Control-Expose-Headers': 'ETag, Link, Location, Retry-After, X-GitHub-OTP, X-RateLimit-Limit, X-RateLimit-Remaining, X-RateLimit-Used, X-RateLimit-Resource, X-RateLimit-Reset, X-OAuth-Scopes, X-Accepted-OAuth-Scopes, X-Poll-Interval, X-GitHub-Media-Type, X-GitHub-SSO, X-GitHub-Request-Id, Deprecation, Sunset, Warning', 'Access-Control-Allow-Origin': '*', 'Strict-Transport-Security': 'max-age=31536000; includeSubdomains; preload', 'X-Frame-Options': 'deny', 'X-Content-Type-Options': 'nosniff', 'X-XSS-Protection': '0', 'Referrer-Policy': 'origin-when-cross-origin, strict-origin-when-cross-origin', 'Content-Security-Policy': "default-src 'none'", 'Server': 'github.com', 'Content-Type': 'application/json; ch

The .headers attribute returns a dictionary-like object, allowing you to access header values by key. For example, to see the content type of the response payload, you can access "Content-Type":

In [12]:
response.headers["Content-Type"]

'application/json; charset=utf-8'

There’s something special about this dictionary-like headers object. The HTTP specification defines headers as case-insensitive, which means you can access them without worrying about their capitalization:

In [13]:
response.headers["content-type"]

'application/json; charset=utf-8'

Whether you use the key "content-type" or "Content-Type", you’ll get the same value.

Now that you’ve seen the most useful attributes and methods of Response in action, you already have a good overview of Requests’ basic usage. You can get content from the internet and work with the response that you receive.

But there’s more to the internet than plain, straightforward URLs. In the next section, you’ll take a step back and see how your responses change when you customize your GET requests to account for query string parameters.



## Add Query String Parameters

One common way to customize a GET request is to pass values through query string parameters in the URL. To do this using get(), you pass data to params. For example, you can use GitHub’s repository search API to look for popular Python repositories:

In [14]:
import requests

response = requests.get(
    "https://api.github.com/search/repositories",
    params={"q": "language:python", "sort": "stars", "order": "desc"},
)

json_response = response.json()
popular_repositories = json_response["items"]
for repo in popular_repositories[:3]:
    print(f"Name: {repo['name']}")
    print(f"Description: {repo['description']}")
    print(f"Stars: {repo['stargazers_count']}\n")

Name: public-apis
Description: A collective list of free APIs
Stars: 478019

Name: free-programming-books
Description: :books: Freely available programming books
Stars: 396347

Name: system-design-primer
Description: Learn how to design large-scale systems. Prep for the system design interview.  Includes Anki flashcards.
Stars: 368978



By passing a dictionary to the params parameter of get(), you’re able to modify the results that come back from the search API.

You can pass params to get() either as a dictionary, as you’ve just done, or as a list of tuples:

In [15]:
import requests

requests.get(
    "https://api.github.com/search/repositories",
    [("q", "language:python"), ("sort", "stars"), ("order", "desc")],
)

<Response [200]>

You can even pass the values as bytes:

In [17]:
requests.get(
    "https://api.github.com/search/repositories",
    params=b"q=language:python&sort=stars&order=desc",
)

<Response [200]>

## Customize Request Headers 

To customize headers, you pass a dictionary of HTTP headers to get() using the headers parameter. For example, you can change your previous search request to highlight matching search terms in the results by specifying the text-match media type in the Accept header:

In [22]:
import requests

response = requests.get(
    "https://api.github.com/search/repositories",
    params={"q": '"real python"'},
    headers={"Accept": "application/vnd.github.text-match+json"},
)

json_response = response.json()
first_repository = json_response["items"][0]
print(first_repository["text_matches"][0]["matches"])

[{'text': 'Real Python', 'indices': [23, 34]}]


## Use Other HTTP Methods

Aside from GET, other popular HTTP methods include POST, PUT, DELETE, HEAD, PATCH, and OPTIONS. For each of these HTTP methods, Requests provides a function with a similar signature to get()

In [23]:
requests.get("https://httpbin.org/get")


<Response [200]>

In [24]:
requests.post("https://httpbin.org/post", data={"key": "value"})

<Response [200]>

In [25]:
requests.put("https://httpbin.org/put", data={"key": "value"})

<Response [200]>

In [26]:
requests.delete("https://httpbin.org/delete")

<Response [200]>

In [27]:
requests.head("https://httpbin.org/get")

<Response [200]>

In [28]:
requests.patch("https://httpbin.org/patch", data={"key": "value"})

<Response [200]>

In [29]:
requests.options("https://httpbin.org/get")

<Response [200]>

In the example above, you called each function to make a request to the httpbin service using the corresponding HTTP method.

All of these functions are high-level shortcuts to requests.request(), which takes the method name as its first argument:

In [30]:
requests.request("GET", "https://httpbin.org/get")

<Response [200]>